<a href="https://colab.research.google.com/github/ajnl/ColabExtractor_Extension/blob/main/01272026_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install ultralytics roboflow pandas onnxruntime


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 120.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 134.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.7 MB/s eta 0:00:00


In [3]:
from roboflow import Roboflow
import os, pandas as pd
from ultralytics import YOLO

# --- Roboflow download ---
rf = Roboflow(api_key="7TQjINczo2Lqo5YpBt40")
project = rf.workspace("ws1-0s0rn").project("pothole-detection-ojrqq-u6yau")
version = project.version(2)  # change
dataset = version.download("yolov8")

data_yaml = os.path.join(dataset.location, "data.yaml")
print("data.yaml:", data_yaml)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Pothole-Detection-2 in yolov8:: 100%|██████████| 6808/6808 [00:00<00:00, 8207.62it/s]

data.yaml: /content/Pothole-Detection-2/data.yaml


In [6]:
# ===============================
# INSTALL DEPENDENCIES (Colab)
# ===============================
!pip install -U ultralytics tensorflow

# ===============================
# IMPORTS
# ===============================
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import time, os

# ===============================
# CONFIG
# ===============================
data_yaml = "/content/Pothole-Detection-2/data.yaml"   # <-- CHANGE THIS
model_name = "yolo11n.pt"
project_name = "pothole-yolo-compare"
run_name = "yolo11n"

epochs = 50
imgsz = 640
batch = 16
device = 0  # GPU

# ===============================
# TRAIN
# ===============================
print("\n==============================")
print("Training YOLOv11n")
print("==============================")

model = YOLO(model_name)

t0 = time.time()
model.train(
    data=data_yaml,
    epochs=epochs,
    imgsz=imgsz,
    batch=batch,
    patience=10,
    device=device,
    project=project_name,
    name=run_name,
    exist_ok=True
)
train_time = time.time() - t0

# ===============================
# VALIDATE BEST MODEL
# ===============================
run_dir = Path("/content/runs/detect") / project_name / run_name
best_pt = run_dir / "weights" / "best.pt"

val_model = YOLO(str(best_pt))
val_res = val_model.val(
    data=data_yaml,
    imgsz=imgsz,
    device=device
)

map5095 = float(val_res.box.map)
map50 = float(val_res.box.map50)

# ===============================
# EXPORT → TFLITE ONLY
# ===============================

# FP32 TFLite
val_model.export(
    format="tflite",
    nms=True
)

# OPTIONAL: INT8 TFLite (smaller & faster)
val_model.export(
    format="tflite",
    int8=True,
    data=data_yaml
)

# ===============================
# FILE SIZES
# ===============================
best_tflite = best_pt.with_suffix(".tflite")
best_int8 = best_pt.with_name("best_int8.tflite")

rows = [{
    "model": "yolo11n",
    "mAP50-95": map5095,
    "mAP50": map50,
    "train_time_s": round(train_time, 1),
    "pt_MB": round(os.path.getsize(best_pt)/(1024*1024), 2),
    "tflite_MB": round(os.path.getsize(best_tflite)/(1024*1024), 2),
    "tflite_int8_MB": round(os.path.getsize(best_int8)/(1024*1024), 2),
}]

df = pd.DataFrame(rows)
df


Training YOLOv11n
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Pothole-Detection-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, p

Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 


ONNX: slimming with onnxslim 0.1.82...
ONNX: export success ✅ 2.5s, saved as '/content/runs/detect/pothole-yolo-compare/yolo11n/weights/best.onnx' (10.2 MB)
Unzipping calibration_image_sample_data_20x128x128x3_float32.npy.zip to /content/calibration_image_sample_data_20x128x128x3_float32.npy...: 100% ━━━━━━━━━━━━ 1/1 43.1files/s 0.0s
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at '/content/runs/detect/pothole-yolo-compare/yolo11n/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 300, 6), dtype=tf.float32, name=None)
Captures:
  135548690231184: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  135548690229648: TensorSpec(shape=(3, 3, 3, 16), dtype=tf.float32, name=None)
  135548690230416: TensorSpec(shape=(16,), dtype=tf.float32, name=None)
  135548690235216: Tensor

FileNotFoundError: [Errno 2] No such file or directory: '/content/runs/detect/pothole-yolo-compare/yolo11n/weights/best.tflite'

In [7]:
# ===============================
# EXPORT → TFLITE (FP32)
# ===============================
fp32_path = val_model.export(
    format="tflite",
    imgsz=imgsz,
    nms=True
)

# ===============================
# EXPORT → TFLITE (INT8 – CPU)
# ===============================
int8_path = val_model.export(
    format="tflite",
    imgsz=imgsz,
    int8=True,
    data=data_yaml,
    device="cpu"
)

print("FP32 TFLite saved to:", fp32_path)
print("INT8 TFLite saved to:", int8_path)


Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)

PyTorch: starting from '/content/runs/detect/pothole-yolo-compare/yolo11n/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.2 MB)

TensorFlow SavedModel: starting export with tensorflow 2.20.0...

ONNX: starting export with onnx 1.20.1 opset 22...
ONNX: slimming with onnxslim 0.1.82...
ONNX: export success ✅ 1.7s, saved as '/content/runs/detect/pothole-yolo-compare/yolo11n/weights/best.onnx' (10.2 MB)
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at '/content/runs/detect/pothole-yolo-compare/yolo11n/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 300, 6), dtype=tf.float32, name=None)
Captures:
  135548464656080: TensorSpec(shape=(4, 2), dtype=tf.

In [8]:
rows = [{
    "model": "YOLOv11n",
    "mAP50-95": round(map5095, 4),
    "mAP50": round(map50, 4),
    "train_time_s": round(train_time, 1),
    "pt_MB": round(os.path.getsize(best_pt)/(1024*1024), 2),
    "tflite_FP32_MB": round(os.path.getsize(fp32_path)/(1024*1024), 2),
    "tflite_INT8_MB": round(os.path.getsize(int8_path)/(1024*1024), 2),
}]

df = pd.DataFrame(rows)
df


,model,mAP50-95,mAP50,train_time_s,pt_MB,tflite_FP32_MB,tflite_INT8_MB
0,YOLOv11n,0.1834,0.3984,2906.1,5.22,10.12,2.85


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import time, os

# Compare starting from YOLOv5
models_to_test = [
    "yolov5n.pt", "yolov5s.pt",
    "yolov8n.pt", "yolov8s.pt",
    "yolov10n.pt","yolov10s.pt",
    "yolo11n.pt", "yolo11s.pt",
    "yolo12n.pt", "yolo12s.pt",
]

# If you want higher versions too, uncomment:
# models_to_test += ["yolov8m.pt","yolov8l.pt","yolov8x.pt",
#                    "yolov10m.pt","yolov10l.pt","yolov10x.pt",
#                    "yolo11m.pt","yolo11l.pt","yolo11x.pt",
#                    "yolo12m.pt","yolo12l.pt","yolo12x.pt"]

base_project = "pothole-yolo-compare"
rows = []

for w in models_to_test:
    print("\n==============================")
    print("Training:", w)
    print("==============================")

    # 1) Train
    model = YOLO(w)

    t0 = time.time()
    model.train(
        data=data_yaml,
        epochs=50,       # keep constant for fair comparison
        imgsz=640,
        batch=16,        # reduce if you get CUDA OOM (try 8 or 4)
        patience=10,
        device=0,        # GPU
        project=base_project,
        name=w.replace(".pt",""),
        exist_ok=True
    )
    train_time = time.time() - t0

    # 2) Validate using best.pt from this run
    run_dir = Path("/content/runs/detect") / base_project / w.replace(".pt","") # Corrected path
    best_pt = run_dir / "weights" / "best.pt"

    val_model = YOLO(str(best_pt))
    val_res = val_model.val(data=data_yaml, imgsz=640, device=0)

    map5095 = float(val_res.box.map)     # mAP50-95
    map50   = float(val_res.box.map50)   # mAP50

    # 3) Export ONNX
    # For deployment convenience, nms=True is often easiest.
    val_model.export(format="onnx", nms=True)
    best_onnx = str(best_pt).replace(".pt", ".onnx")

    # Sizes
    pt_mb = os.path.getsize(best_pt)/(1024*1024)
    onnx_mb = os.path.getsize(best_onnx)/(1024*1024)

    rows.append({
        "model": w,
        "mAP50-95": map5095,
        "mAP50": map50,
        "train_time_s": round(train_time, 1),
        "best_pt": str(best_pt),
        "best_onnx": best_onnx,
        "pt_MB": round(pt_mb, 2),
        "onnx_MB": round(onnx_mb, 2),
    })

df = pd.DataFrame(rows).sort_values("mAP50-95", ascending=False)
df


Training: yolov5n.pt
PRO TIP 💡 Replace 'model=yolov5n.pt' with new 'model=yolov5nu.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.

Ultralytics 8.4.8 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Pothole-Detection-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=Fals

Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 


ONNX: slimming with onnxslim 0.1.82...
ONNX: export success ✅ 9.0s, saved as '/content/runs/detect/pothole-yolo-compare/yolov5n/weights/best.onnx' (9.8 MB)

Export complete (9.3s)
Results saved to /content/runs/detect/pothole-yolo-compare/yolov5n/weights
Predict:         yolo predict task=detect model=/content/runs/detect/pothole-yolo-compare/yolov5n/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/runs/detect/pothole-yolo-compare/yolov5n/weights/best.onnx imgsz=640 data=/content/Pothole-Detection-2/data.yaml  
Visualize:       https://netron.app

Training: yolov5s.pt
PRO TIP 💡 Replace 'model=yolov5s.pt' with new 'model=yolov5su.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.

Ultralytics 8.4.8 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.

Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 


ONNX: slimming with onnxslim 0.1.82...
ONNX: export success ✅ 2.5s, saved as '/content/runs/detect/pothole-yolo-compare/yolov5s/weights/best.onnx' (35.0 MB)

Export complete (3.6s)
Results saved to /content/runs/detect/pothole-yolo-compare/yolov5s/weights
Predict:         yolo predict task=detect model=/content/runs/detect/pothole-yolo-compare/yolov5s/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/runs/detect/pothole-yolo-compare/yolov5s/weights/best.onnx imgsz=640 data=/content/Pothole-Detection-2/data.yaml  
Visualize:       https://netron.app

Training: yolov8n.pt
Ultralytics 8.4.8 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Pothole-Detection

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# YOLO families with both n and s variants
families = ["YOLOv5", "YOLOv8", "YOLOv10", "YOLO11", "YOLO12"]

# Values extracted from your table
map5095_n = [0.109354571, 0.129594241, 0.144498743, 0.147722380, 0.126266917]
map5095_s = [0.123791821, 0.133395296, 0.147511399, 0.124399616, 0.125687039]

map50_n   = [0.277185725, 0.335584729, 0.338256913, 0.361951394, 0.342705901]
map50_s   = [0.315735905, 0.321866447, 0.344644470, 0.337866294, 0.297975189]

ptmb_n    = [5.02, 5.96, 5.49, 5.22, 5.26]
ptmb_s    = [17.66, 21.47, 15.76, 18.29, 18.05]

def dumbbell(ax, left, right, ylabels, xlabel, title):
    y = np.arange(len(ylabels))
    ax.hlines(y, left, right, linewidth=2)
    ax.scatter(left,  y, s=60, label="n (lightweight)")
    ax.scatter(right, y, s=60, label="s (standard)")
    ax.set_yticks(y)
    ax.set_yticklabels(ylabels)
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    ax.grid(True, axis="x", linestyle="--", alpha=0.5)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

dumbbell(
    axes[0],
    map5095_n, map5095_s, families,
    "mAP@50–95",
    "Detection Accuracy (mAP@50–95)"
)

dumbbell(
    axes[1],
    map50_n, map50_s, families,
    "mAP@50",
    "Detection Accuracy (mAP@50)"
)

dumbbell(
    axes[2],
    ptmb_n, ptmb_s, families,
    "Model Size (PT, MB)",
    "Model Size Comparison"
)

# Single legend for the entire figure
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)

plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.savefig("figure_complete_yolo_family_n_vs_s.png", dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

models = [
    "YOLO11-n", "YOLOv10-s", "YOLOv10-n", "YOLOv8-s", "YOLOv8-n",
    "YOLO12-n", "YOLO12-s", "YOLO11-s", "YOLOv5-s", "YOLOv5-n"
]

training_time = [
    551.4, 491.3, 650.6, 279.8, 248.7,
    268.3, 328.9, 292.9, 201.5, 251.7
]

plt.figure(figsize=(10, 5))
bars = plt.bar(models, training_time)

plt.xlabel("YOLO Models")
plt.ylabel("Training Time (seconds)")
plt.title("Training Time Comparison of YOLO Models")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.5)

# Add value labels on top of bars
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 8,                     # small offset above bar
        f"{height:.1f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
plt.savefig("training_time_comparison_with_values.png", dpi=300)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

models = [
    "YOLO11-n", "YOLOv10-s", "YOLOv10-n", "YOLOv8-s", "YOLOv8-n",
    "YOLO12-n", "YOLO12-s", "YOLO11-s", "YOLOv5-s", "YOLOv5-n"
]

pt_size = [
    5.22, 15.76, 5.49, 21.47, 5.96,
    5.26, 18.05, 18.29, 17.66, 5.02
]

onnx_size = [
    10.14, 27.80, 8.91, 42.70, 11.72,
    10.12, 35.58, 36.20, 35.03, 9.82
]

x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(11, 5))
bars1 = plt.bar(x - width/2, pt_size, width, label="PyTorch (PT)")
bars2 = plt.bar(x + width/2, onnx_size, width, label="ONNX")

plt.xticks(x, models, rotation=45, ha="right")
plt.ylabel("Model Size (MB)")
plt.xlabel("YOLO Models")
plt.title("Model Size Comparison for Deployment")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2,
            height + 0.6,
            f"{height:.2f}",
            ha="center",
            va="bottom",
            fontsize=8
        )

plt.tight_layout()
plt.savefig("model_size_comparison.png", dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = [
    "YOLO11-n", "YOLOv10-s", "YOLOv10-n", "YOLOv8-s", "YOLOv8-n",
    "YOLO12-n", "YOLO12-s", "YOLO11-s", "YOLOv5-s", "YOLOv5-n"
]

map_50_95 = np.array([
    0.14772238, 0.147511399, 0.144498743, 0.133395296, 0.129594241,
    0.126266917, 0.125687039, 0.124399616, 0.123791821, 0.109354571
])

# Sort descending for ranking
idx = np.argsort(map_50_95)[::-1]
models_sorted = [models[i] for i in idx]
map_sorted = map_50_95[idx]

plt.figure(figsize=(10, 5))
bars = plt.bar(models_sorted, map_sorted)

plt.xlabel("YOLO Models")
plt.ylabel("mAP@50–95")
plt.title("Visual Comparison of YOLO Model Performance (mAP@50–95)")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.5)

# Add value labels
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, h + 0.002, f"{h:.4f}",
             ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("figure_model_performance_map50_95_ranked.png", dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

models = [
    "YOLO11-n", "YOLOv10-s", "YOLOv10-n", "YOLOv8-s", "YOLOv8-n",
    "YOLO12-n", "YOLO12-s", "YOLO11-s", "YOLOv5-s", "YOLOv5-n"
]

pt_size = [
    5.22, 15.76, 5.49, 21.47, 5.96,
    5.26, 18.05, 18.29, 17.66, 5.02
]

map_50_95 = [
    0.14772238, 0.147511399, 0.144498743, 0.133395296, 0.129594241,
    0.126266917, 0.125687039, 0.124399616, 0.123791821, 0.109354571
]

plt.figure(figsize=(7, 5))
plt.scatter(pt_size, map_50_95)

for i, m in enumerate(models):
    plt.annotate(m, (pt_size[i] + 0.2, map_50_95[i]), fontsize=8)

plt.xlabel("Model Size (PT, MB)")
plt.ylabel("mAP@50–95")
plt.title("Accuracy–Efficiency Trade-off of YOLO Models")
plt.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("figure_accuracy_vs_model_size.png", dpi=300)
plt.show()


In [ ]:
df.to_csv("yolo_comparison.csv", index=False)
print("Saved yolo_comparison.csv")
